In [1]:
import math
import time
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
from PIL import Image, ImageDraw

import ee
ee.Initialize()

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


### `latlon_to_tile(lat, lon, zoom)`

**What it is:**  
Converts geographic coordinates (latitude/longitude) to map tile coordinates (x, y) at a specific zoom level.

**How it works:**  
Uses the standard Web Mercator projection formulas. The longitude is linearly mapped to the x coordinate, while the latitude uses a more complex mathematical transformation involving tangent and natural logarithm to account for the Mercator projection's distortion (which stretches areas near the poles). The `2^zoom` factor determines how many tiles exist at that zoom level.

**Why we use it:**  
Map services like OpenTopoMap organize their imagery into a grid of tiles identified by x, y, and zoom. To download the correct tiles for a specific location, we need to know which tiles contain that point. This function gives us the exact tile coordinates to request from the server.

In [4]:
# --- Lat/lon <-> tile number conversion (standard Web Mercator formula) ---
def latlon_to_tile(lat, lon, zoom):
    """Convert latitude/longitude to tile coordinates at a given zoom level."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

print(latlon_to_tile(40.7128, -74.0060, 13))

(2411, 3080)


### `latlon_to_pixel(lat, lon, zoom, tile_size=256)`

**What it is:**  
Converts geographic coordinates to absolute pixel coordinates within the entire world map at a given zoom level.

**How it works:**  
Similar to `latlon_to_tile`, but instead of returning tile indices, it multiplies the result by the tile size (256 pixels) to get the exact pixel position in the global coordinate space. This gives a continuous position rather than discrete tile numbers.

**Why we use it:**  
When we have multiple tiles stitched together on a canvas, we need to know exactly where to place a marker within that canvas. Tile coordinates alone would only tell us which tile the point is on, not where within the tile. Pixel coordinates give us the precision to place markers exactly at the geographic location, even at the sub-tile level.

In [5]:
def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convert lat/lon to absolute PIXEL coordinates (not just tile) at the given zoom level.
    This allows locating the exact point within the canvas, not just the tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    # Calculate pixel coordinates in the global pixel space
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel

print(latlon_to_pixel(40.7128, -74.0060, 1))

(150.74702222222223, 192.50216870810044)


### `draw_marker(canvas, x, y, radius=8, color=(220, 30, 30))`

**What it is:**  
Draws a circular marker (like a pin drop) at a specific pixel position on the map canvas.

**How it works:**  
Creates a drawing context for the PIL image canvas, then draws an ellipse (circle) centered at the given (x, y) coordinates. The marker has a red fill with a white outline for visibility against various map backgrounds.

**Why we use it:**  
To visually indicate the exact location of the point of interest on the final map image. Without this marker, it would be impossible to know where the specified coordinates are located within the generated map.


In [8]:
def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Draw a circular marker with white outline at position (x, y) on the canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )

### `get_point_map(lat, lon, zoom=17, radius=2, out_prefix="point_map", show_marker=True)`

**What it is:**  
The main function that generates a composite map image centered on a specific geographic point by downloading, stitching, and annotating map tiles.

**How it works:**

1. Calculates which tile contains the center point
    
2. Determines a grid of tiles around that center based on the radius parameter
    
3. Creates a blank canvas large enough to hold all tiles
    
4. Downloads each tile from OpenTopoMap's tile server with rate limiting (0.5 second delay between requests)
    
5. Pastes each downloaded tile onto the canvas at the correct position
    
6. Optionally draws a marker at the exact geographic coordinates
    
7. Saves the final image with a timestamp in the filename
    

**Why we use it:**  
To create localized, high-resolution topographic maps for specific points of interest without needing to download or process entire regions. This is useful for site-specific analysis, field work planning, or generating map figures for reports where only a small area around a point is relevant. The function handles all the complexity of coordinate conversion, tile downloading, rate limiting, and image composition automatically.

In [13]:
def get_point_map(lat, lon, zoom=17, radius=2, out_prefix="point_map", show_marker=True):
    """
    Generate a map centered on a specific point by downloading and stitching map tiles.
    
    Parameters:
        lat, lon: coordinates of the center point
        zoom: zoom level (OpenTopoMap supports up to 17)
        radius: how many tiles to add around the center in each direction.
                radius=2 -> 5x5 tile grid
        show_marker: if True, draws a red dot at the exact location
        out_prefix: base name for the output file; automatically appended with -vYYMMDDHHMMSS.png
    
    Returns:
        str: path to the saved image file
    """
    # Generate timestamp at the moment of map creation: YYMMDDHHMMSS (e.g., 260803143022)
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    out_path = f"{out_prefix}-v{timestamp}.png"

    # Calculate the tile coordinates for the center point
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    
    # Define the range of tiles to download (creating a grid around the center)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    # Calculate the total canvas size based on the tile grid
    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    # Set a user agent header to identify our application to the tile server
    headers = {"User-Agent": "agri_land_suitability_pipeline (your_email@example.com)"}

    # Download each tile in the grid
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            # Construct the URL for the tile from OpenTopoMap
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            # Handle failed tile downloads
            if resp.status_code != 200:
                print(f"Tile {x},{y} failed with status {resp.status_code}")
                time.sleep(0.5)
                continue

            # Try to open and paste the tile onto the canvas
            try:
                tile_img = Image.open(BytesIO(resp.content))
                # Calculate the position where this tile should be placed on the canvas
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} failed: {e}")

            # Respect the rate limit (~2 requests/second max)
            time.sleep(0.5)

    # Draw a marker at the exact point location if requested
    if show_marker:
        # Get the absolute pixel position of the point in the global pixel space
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        # Convert to canvas-relative coordinates by subtracting the canvas origin
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        # Draw the marker at the calculated position
        draw_marker(canvas, px_canvas, py_canvas)

    # Save the final composite image
    canvas.save(out_path)
    print(f"Map saved to {out_path} ({width}x{height}px)")
    return out_path


# Test the function with example coordinates
get_point_map(7.4584221918243045, -73.222052853104,out_prefix="El_Playon", zoom=16, radius=2)
# Sugarcane 3.580109040361371, -76.31299479308868 Colombia
#Location: {'city': 'Bundaberg', 'country': 'Australia'}(-24.8660, 152.3489)

Map saved to El_Playon-v260803200622.png (1280x1280px)


'El_Playon-v260803200622.png'

In [2]:
import math
import time
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
from PIL import Image, ImageDraw


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convierte lat/lon a coordenadas de PIXEL absolutas (no solo tile) en el zoom dado.
    Esto es lo que permite ubicar el punto exacto dentro del canvas, no solo el tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Dibuja un marcador circular con contorno blanco en la posición (x, y) del canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_point_map(lat, lon, zoom=17, radius=2, out_prefix="point_map",
                   output_dir="../img/maps", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    out_prefix: nombre base del archivo; se le agrega -vYYMMDDHHMMSS.png automáticamente
    output_dir: carpeta donde se guarda la imagen (relativa al directorio desde
                donde corres el script/notebook, o una ruta absoluta). Se crea
                sola si no existe.
    """
    # Timestamp al momento de generar el mapa: YYMMDDHHMMSS (ej: 260803143022)
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    if show_marker:
        # Pixel absoluto del punto en el mundo, menos el origen del canvas (esquina x_min,y_min)
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba

# Test the function with example coordinates
get_point_map(7.4584221918243045, -73.222052853104,out_prefix="El_Playon", zoom=16, radius=2)
# Sugarcane 3.580109040361371, -76.31299479308868 Colombia
#Location: {'city': 'Bundaberg', 'country': 'Australia'}(-24.8660, 152.3489)

Mapa guardado en ../img/maps/El_Playon-v260804185205.png (1280x1280px)


PosixPath('../img/maps/El_Playon-v260804185205.png')

In [4]:
import math
import time
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
from PIL import Image, ImageDraw


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convierte lat/lon a coordenadas de PIXEL absolutas (no solo tile) en el zoom dado."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(255, 40, 40)):
    """Dibuja un marcador circular con contorno blanco en la posición (x, y) del canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Descargar y pegar tiles de imagen satelital (ESRI World Imagery) alrededor de un punto ---
def get_point_satellite_image(lat, lon, zoom=18, radius=2, out_prefix="point_sat",
                               output_dir="../img/maps", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom. ESRI World Imagery suele llegar hasta ~19-20 en zonas
          bien cubiertas, pero varía según la región (en zonas rurales puede
          quedarse en 16-17 antes de verse borroso). Probá con 18 primero.
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    out_prefix: nombre base del archivo; se le agrega -vYYMMDDHHMMSS.png
    output_dir: carpeta donde se guarda la imagen (se crea sola si no existe)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email_real@dominio.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            # OJO: ESRI usa el orden {z}/{y}/{x}, al revés de OpenTopoMap ({z}/{x}/{y})
            url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y}/{x}"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.2)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.2)  # ESRI no publica un límite tan estricto como OpenTopoMap, pero igual conviene ser prudente

    if show_marker:
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Imagen guardada en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
# Test the function with example coordinates
get_point_satellite_image(7.4584221918243045, -73.222052853104,out_prefix="El_Playon_sat", zoom=16, radius=2)
# Sugarcane 3.580109040361371, -76.31299479308868 Colombia
#Location: {'city': 'Bundaberg', 'country': 'Australia'}(-24.8660, 152.3489)

Imagen guardada en ../img/maps/El_Playon_sat-v260804190225.png (1280x1280px)


PosixPath('../img/maps/El_Playon_sat-v260804190225.png')

In [ ]:
## MDS650_v260804_Point_Map.ipynb

import time
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
from PIL import Image
import math

from tile_utils import latlon_to_tile, latlon_to_pixel, draw_marker, draw_area_box


# Configuración por proveedor: URL template, orden de x/y, zoom máximo típico,
# y pausa recomendada entre requests (cada servicio tiene su propia política de uso).
PROVIDERS = {
    "opentopomap": {
        "url_template": "https://tile.opentopomap.org/{z}/{x}/{y}.png",
        "max_zoom": 17,
        "sleep": 0.5,
    },
    "esri": {
        # OJO: ESRI usa el orden {z}/{y}/{x}, al revés del estándar {z}/{x}/{y}
        "url_template": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        "max_zoom": 17,
        "sleep": 0.2,
    },
}


def get_point_map(lat, lon, area_meters=56, provider="esri", zoom=None, radius=2, 
                  out_prefix=None, output_dir="../img/maps", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    provider: "esri" (foto satelital real) u "opentopomap" (mapa topográfico/altitud)
    zoom: si no se especifica, usa el max_zoom típico del proveedor
    radius: cuántos tiles agregar alrededor del centro en cada dirección
    out_prefix: nombre base del archivo; si no se especifica, usa el nombre del proveedor
    output_dir: carpeta donde se guarda la imagen (se crea sola si no existe)
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    """
    if provider not in PROVIDERS:
        raise ValueError(f"Proveedor '{provider}' no reconocido. Opciones: {list(PROVIDERS.keys())}")

    config = PROVIDERS[provider]
    zoom = zoom or config["max_zoom"]
    out_prefix = out_prefix or f"point_{provider}"

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email_real@dominio.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = config["url_template"].format(z=zoom, x=x, y=y)
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(config["sleep"])
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(config["sleep"])

    if show_marker:
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)
        draw_area_box(canvas, px_canvas, py_canvas, area_meters, lat=lat, zoom=zoom)

    canvas.save(out_path)
    print(f"Imagen guardada en {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Foto satelital real
    ha = 2
    area_meters = (math.sqrt(ha*10000))/2
    Latitude=7.4584221918243045
    Longitude=-73.222052853104
    get_point_map(Latitude, Longitude, area_meters, provider="esri", radius=2)

    # Mapa de altitud/relieve (si lo necesitas de nuevo más adelante)
    get_point_map(Latitude, Longitude, area_meters, provider="opentopomap", radius=2)

# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
